In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\14037\AppData\Local\Programs\Python\Python310\python.exe
3.10.9 (tags/v3.10.9:1dd9be6, Dec  6 2022, 20:01:21) [MSC v.1934 64 bit (AMD64)]


In [3]:
import numpy as np
import pandas as pd
import os

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# ==========================
# 配置区域
# ==========================

INPUT_FILE = r"D:\萝卜投稿专用\Original Research - Spinal Infection Following Internal Fixation Surgery\插值选择\训练集_原始数据.xlsx"

OUTPUT_DIR = r"D:\萝卜投稿专用\Original Research - Spinal Infection Following Internal Fixation Surgery\插值选择\MICE_RF.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================
# 读取数据
# ==========================

print("读取数据...")

data = pd.read_excel(INPUT_FILE)

if "Infection" not in data.columns:
    raise ValueError("缺少 Infection 列")

# 保存 Infection 标签；插补和参数搜索时不把 Infection 作为预测因子
infection_label = data["Infection"].copy()
feature_cols = [
    col for col in data.columns
    if col != "Infection"
]
feature_data = data[feature_cols].copy()

# ==========================
# 找完整列
# ==========================

complete_cols = [
    col for col in feature_data.columns
    if feature_data[col].isnull().sum() == 0
]

print("\n完整列:")
print(complete_cols)

# ==========================
# 制造10%缺失
# ==========================

np.random.seed(42)

data_with_missing = feature_data.copy()

missing_records = pd.DataFrame()

for col in complete_cols:

    missing_idx = np.random.choice(
        feature_data.index,
        size=int(len(feature_data) * 0.10),
        replace=False
    )

    missing_records = pd.concat([
        missing_records,
        pd.DataFrame({
            "列名": col,
            "行索引": missing_idx,
            "真实值": feature_data.loc[missing_idx, col]
        })
    ])

    data_with_missing.loc[
        missing_idx,
        col
    ] = np.nan

print("\n已完成人工缺失")

读取数据...

完整列:
['L1', 'L1%', 'M1', 'M%1', 'N1', 'N%1', 'PLT1', 'WBC1', 'NLR1', 'PLR1', 'Age', 'Operating time（h）']

已完成人工缺失


In [4]:
# 开始30次MICE

n_estimators_list = [
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100
]
max_depth_list = [5, 10, None]

phase1_results = []

for n_est in n_estimators_list:
    for max_depth in max_depth_list:

        estimator = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1
        )

    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=15,
        random_state=42
    )

    imputed_data = imputer.fit_transform(
        data_with_missing
    )

    imputed_df = pd.DataFrame(
        imputed_data,
        columns=feature_cols
    )

    # 恢复 Infection 标签，但它不参与插补
    imputed_df["Infection"] = infection_label.values

    nrmse_list = []

    for col in complete_cols:

        col_records = missing_records[
            missing_records["列名"] == col
        ]

        y_true = col_records["真实值"]

        y_pred = imputed_df.loc[
            col_records["行索引"],
            col
        ]

        rmse = np.sqrt(
            np.mean(
                (y_true - y_pred) ** 2
            )
        )

        nrmse = rmse / (
            y_true.max() - y_true.min()
        )

        nrmse_list.append(nrmse)

    mean_nrmse = np.mean(
        nrmse_list
    )

    sd_nrmse = np.std(
        nrmse_list,
        ddof=1
    )

    phase1_results.append({

        "n_estimators": n_est,
        "max_depth": max_depth,

        "Mean_NRMSE": mean_nrmse,
        "SD_NRMSE": sd_nrmse
    })

    print(
        f"Mean_NRMSE={mean_nrmse:.5f}, "
        f"SD_NRMSE={sd_nrmse:.5f}"
    )

phase1_df = pd.DataFrame(
    phase1_results
)

phase1_df = phase1_df.sort_values(
    "Mean_NRMSE"
)

print("\nTOP3")

print(
    phase1_df.head(3)
)


KeyboardInterrupt: 

In [ ]:
# Phase 2
# 对 Phase 1 最好的3组参数进行多随机种子稳定性验证


top3_params = [
    {
        "n_estimators": 10,
        "max_depth": 5
    },
    {
        "n_estimators": 20,
        "max_depth": 10
    },
    {
        "n_estimators": 30,
        "max_depth": None
    }
]

phase2_results = []

# 10个随机种子
seed_list = list(range(100, 1100, 100))

for params in top3_params:

    n_est = params["n_estimators"]
    max_depth = params["max_depth"]

    print("\n========================================")
    print(
        f"Testing n_estimators={n_est}, "
        f"max_depth={max_depth}"
    )
    print("========================================")

    run_nrmse = []

    for seed in seed_list:

        print(f"Seed={seed}")

        imputer = IterativeImputer(

            estimator=RandomForestRegressor(

                n_estimators=n_est,

                max_depth=max_depth,

                random_state=seed,

                n_jobs=-1
            ),

            max_iter=15,

            random_state=seed
        )

        imputed_data = imputer.fit_transform(
            data_with_missing
        )

        imputed_df = pd.DataFrame(
            imputed_data,
            columns=feature_cols
        )

        # 恢复 Infection 标签
        # Infection 不参与插补
        imputed_df["Infection"] = infection_label.values

        nrmse_list = []

        for col in complete_cols:

            col_records = missing_records[
                missing_records["列名"] == col
            ]

            # 如果该列没有人工缺失记录，则跳过
            if len(col_records) == 0:
                continue

            y_true = col_records["真实值"].astype(float)

            y_pred = imputed_df.loc[
                col_records["行索引"],
                col
            ].astype(float)

            rmse = np.sqrt(
                np.mean(
                    (y_true - y_pred) ** 2
                )
            )

            value_range = (
                y_true.max() - y_true.min()
            )

            # 防止真实值极差为0导致除零
            if value_range == 0:
                continue

            nrmse = rmse / value_range

            nrmse_list.append(
                nrmse
            )

        # 确保有可评价的列
        if len(nrmse_list) == 0:

            mean_nrmse = np.nan

        else:

            mean_nrmse = np.mean(
                nrmse_list
            )

        run_nrmse.append(
            mean_nrmse
        )

    # 去掉可能出现的 NaN
    valid_run_nrmse = np.array(
        run_nrmse,
        dtype=float
    )

    valid_run_nrmse = valid_run_nrmse[
        ~np.isnan(valid_run_nrmse)
    ]

    overall_mean = np.mean(
        valid_run_nrmse
    )

    overall_sd = np.std(
        valid_run_nrmse,
        ddof=1
    )

    ci_low = np.percentile(
        valid_run_nrmse,
        2.5
    )

    ci_high = np.percentile(
        valid_run_nrmse,
        97.5
    )

    phase2_results.append({

        "n_estimators": n_est,

        "max_depth": max_depth,

        "Number_of_Seeds": len(valid_run_nrmse),

        "Mean_NRMSE": overall_mean,

        "SD_NRMSE": overall_sd,

        "CI_Low": ci_low,

        "CI_High": ci_high

    })


# ==========================
# 整理最终结果
# ==========================

phase2_df = pd.DataFrame(
    phase2_results
)

phase2_df = phase2_df.sort_values(
    by="Mean_NRMSE",
    ascending=True
).reset_index(drop=True)

phase2_df.insert(
    0,
    "Rank",
    range(1, len(phase2_df) + 1)
)

phase2_df.to_excel(
    "MICE_RF_Phase2_Final_Ranking.xlsx",
    index=False
)

print("\n========================================")
print("FINAL RESULT")
print("========================================")

print(
    phase2_df.to_string(index=False)
)